# 부록 C 본문이 미룬 코드

『AI 시대의 파이썬 데이터 시각화』 예제 노트북입니다. 셀 첫 줄의 번호(P·C·B)는 책 지면의 코드 번호와 같습니다. 위에서부터 차례로 실행합니다.

## 준비

새 노트북이면 이 장의 코드보다 먼저 아래 준비 셀을 실행합니다(책의 앞 장과 같은 셀).

**코랩이면 이 셀(P02)을, 내 컴퓨터면 다음 셀(P03)을 실행합니다.** 둘 다 실행하지 않습니다.

In [ ]:
# P02 | 한글 글꼴(코랩): 나눔글꼴을 설치하고 Matplotlib에 등록합니다.
!apt-get -qq update
!apt-get -qq install -y fonts-nanum

import matplotlib.pyplot as plt
from matplotlib import font_manager

for path in font_manager.findSystemFonts():
    if 'Nanum' in path:
        font_manager.fontManager.addfont(path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

**내 컴퓨터(Mac·Windows)면 이 셀을 실행합니다.** Windows는 글꼴 이름을 `'Malgun Gothic'`으로 바꿉니다.

In [ ]:
# P03 | 한글 글꼴(Mac·Windows): 그래프를 그리기 전에 한 번 실행합니다.
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'AppleGothic'   # Windows는 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False    # 음수 기호가 깨지지 않게

In [ ]:
# P05 | Plotly 준비: 라이브러리를 불러오고 기본 모양을 정합니다.
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

print('Plotly:', plotly.__version__)
pio.templates.default = 'plotly_white'               # 흰 배경·옅은 격자
px.defaults.line_dash_sequence = ['solid', 'dash']   # 둘째 그룹은 파선
px.defaults.symbol_sequence = ['circle', 'x']        # 둘째 그룹은 X 점

In [ ]:
# C01 | 예제 자료: 10장의 12개월 표를 다시 만들고 월·분기 열을 더합니다.
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns

sales_a = [100, 120, 150, 130, 140, 160, 150, 170, 180, 160, 190, 200]
sales_b = [70, 90, 100, 120, 110, 130, 140, 150, 160, 170, 160, 180]
marketing_b = [7, 9, 11, 12, 11, 13, 14, 15, 16, 17, 16, 18]
df = pd.DataFrame({
    'date': list(pd.date_range('2026-01-01', periods=12, freq='MS')) * 2,
    'branch': ['A'] * 12 + ['B'] * 12,
    'sales': sales_a + sales_b,
    'marketing': [s // 10 for s in sales_a] + marketing_b})
df['month'] = df['date'].dt.month          # 1~12
df['quarter'] = df['date'].dt.quarter      # 1~4
branch_colors = {'A': '#0072B2', 'B': '#D55E00'}   # 지점 색(4장과 같음)
print(len(df), '행 ·', df.groupby('branch')['sales'].sum().to_dict())

## C.1 Matplotlib

In [ ]:
# C02 | 값을 색으로: 점의 색이 월을 나타내고, 색 막대가 그 뜻을 읽게 합니다.
a = df[df['branch'] == 'A']
fig, ax = plt.subplots(figsize=(5.2, 2.3))
sc = ax.scatter(a['marketing'], a['sales'], c=a['month'], cmap='viridis')
cbar = fig.colorbar(sc, ax=ax, ticks=[1, 4, 7, 10, 12])
cbar.ax.set_title('월')                 # 색 막대 위 제목
ax.set(title='A지점 광고비와 매출: 점 색은 월',
       xlabel='광고비 (백만원)', ylabel='매출 (백만원)')
fig.tight_layout()
plt.show()

In [ ]:
# C03 | 날짜·% 눈금과 공통 범례: 두 패널에 범례를 하나만 답니다.
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter, PercentFormatter

total = df.groupby('date')['sales'].sum()
b_rows = df[df['branch'] == 'B'].set_index('date')
b_share = b_rows['sales'] / total * 100
fig, axes = plt.subplots(1, 2, figsize=(6, 2.3), layout='constrained')
for branch, part in df.groupby('branch'):
    axes[0].plot(part['date'], part['sales'],
                 marker='s' if branch == 'B' else 'o',
                 color=branch_colors[branch], label=f'{branch}지점',
                 linestyle='--' if branch == 'B' else '-')
axes[1].plot(b_share.index, b_share, marker='s', linestyle='--',
             color=branch_colors['B'])
month_label = FuncFormatter(lambda x, pos: f'{mdates.num2date(x).month}월')
odd_months = range(1, 13, 2)                     # 1, 3, …, 11월
for ax in axes:
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=odd_months))
    ax.xaxis.set_major_formatter(month_label)
axes[1].yaxis.set_major_formatter(PercentFormatter(xmax=100))
axes[0].set(title='월 매출', ylabel='매출 (백만원)', ylim=(0, 220))
axes[1].set(title='B지점 매출 비중', ylim=(0, 60))
fig.legend(loc='outside upper center', ncols=2)
plt.show()

In [ ]:
# C04 | 색각 이상 배려 팔레트: 팔레트에서 색을 꺼내 집단 색 사전을 만듭니다.
palette = sns.color_palette('colorblind')
group_colors = dict(zip(['A', 'B', 'C'], palette.as_hex()))
print(group_colors)

## C.2 Seaborn

In [ ]:
# C05 | 범주형 패싯: catplot()은 막대 같은 범주형 차트를 지점마다 나눕니다.
g = sns.catplot(data=df, x='quarter', y='marketing', col='branch',
                kind='bar', estimator='sum', errorbar=None, hue='branch',
                palette=branch_colors, legend=False,
                height=2.1, aspect=1.35)
g.set_axis_labels('분기', '분기 광고비 (백만원)')
g.set_titles('{col_name}지점')
plt.show()
print(df.groupby(['branch', 'quarter'])['marketing'].sum().unstack())

In [ ]:
# C06 | 분포 패싯: 8장의 배송 표로 지점마다 히스토그램 패널을 나눕니다.
delivery = pd.DataFrame({
    'branch': ['A'] * 20 + ['B'] * 20,
    'minutes': [18, 19, 19, 20, 20, 20, 21, 21, 22, 20] * 2
             + [8, 10, 12, 14, 16, 18, 20, 22, 25, 55] * 2})
g = sns.displot(data=delivery, x='minutes', col='branch', hue='branch',
                palette=branch_colors, bins=range(0, 61, 5), legend=False,
                height=2.1, aspect=1.35)
g.set_axis_labels('배송시간 (분)', '건수')
g.set_titles('{col_name}지점')
plt.show()

## C.3 Plotly

In [ ]:
# C07 | Plotly 분포와 패싯: 히스토그램을 지점마다 같은 구간으로 나눕니다.
fig_hist = px.histogram(delivery, x='minutes', facet_col='branch',
                        facet_col_spacing=0.08,   # 패널 간격(폭 비율)
                        color='branch', color_discrete_map=branch_colors,
                        labels={'minutes': '배송시간 (분)',
                                'branch': '지점'})
fig_hist.update_traces(xbins=dict(start=0, end=60, size=5),   # 구간 5분
                       marker_line_width=1, marker_line_color='white')
fig_hist.for_each_annotation(
    lambda a: a.update(text=a.text.split('=')[1] + '지점'))
fig_hist.update_layout(showlegend=False, yaxis_title='건수')
fig_hist.show()

In [ ]:
# C08 | 범주 순서와 구간 강조: 순서를 고정하고 하반기·목표선을 그립니다.
branch_dashes = {'A': 'solid', 'B': 'dash'}         # 지점 선 모양(10장)
data = df.sort_values(['branch', 'date'], ascending=[False, True])
fig = px.line(data, x='date', y='sales', color='branch', markers=True,
              category_orders={'branch': ['A', 'B']},
              color_discrete_map=branch_colors,
              line_dash='branch', line_dash_map=branch_dashes,
              labels={'date': '월', 'sales': '매출 (백만원)',
                      'branch': '지점'},
              title='월 매출: 하반기와 목표선')
fig.add_vrect(x0=df['date'][df['month'] == 7].min(), x1=df['date'].max(),
              fillcolor='gray', opacity=0.12, line_width=0)
fig.add_hline(y=150, line_dash='dot', line_color='gray',
              annotation_text='가정한 목표 150',
              annotation_position='bottom right')
fig.update_xaxes(tickformat='%-m월')
fig.show()
print([trace.name for trace in fig.data])

In [ ]:
# C09 | 툴팁 서식과 도구 막대: 날짜를 한국어로 적고 도구 막대를 숨깁니다.
fig.update_traces(hovertemplate='%{x|%-m월}<br>%{y:,}백만원'
                                '<extra>%{fullData.name}지점</extra>')
print(fig.data[0].hovertemplate)
fig.show(config={'displayModeBar': False})

## C.4 Streamlit

지면의 번호 없는 코드(앱·배포 파일에 넣는 조각이나 지면이 말한 자리에서 실행하는 확인 코드 — 참고용):

```python
if st.sidebar.button('CSV 다시 읽기'):     # CSV를 고친 뒤 누름
    load_data.clear()
```

지면의 번호 없는 코드(앱·배포 파일에 넣는 조각이나 지면이 말한 자리에서 실행하는 확인 코드 — 참고용):

```python
if 'branches' not in st.session_state:          # 처음 실행할 때만
    st.session_state['branches'] = ['A', 'B']


def reset_branches():                             # 누르면 재실행보다 먼저
    st.session_state['branches'] = ['A', 'B']


branches = st.sidebar.multiselect('지점', ['A', 'B'], key='branches')
st.sidebar.button('지점 처음대로', on_click=reset_branches)
```

지면의 번호 없는 코드(앱·배포 파일에 넣는 조각이나 지면이 말한 자리에서 실행하는 확인 코드 — 참고용):

```python
if expanded:                                     # 확장도 같은 view로
    monthly = view.groupby('month', as_index=False)[metric].sum()
    cols = ['sales', 'profit']
    period = view.groupby('branch', as_index=False)[cols].sum()
    period['margin'] = period['profit'] / period['sales'] * 100
    month_fig = px.bar(monthly, x='month', y=metric,
                       color_discrete_sequence=['#888888'],
                       labels={'month': '월',
                               metric: f'{metric_label} (백만원)'},
                       title='고른 지점의 월별 합계')
    month_fig.update_xaxes(dtick=1)
    margin_fig = px.bar(period, x='branch', y='margin', color='branch',
                        color_discrete_map=branch_colors,
                        pattern_shape='branch',
                        pattern_shape_map={'A': '', 'B': '/'},
                        labels={'branch': '지점',
                                'margin': '영업이익률 (%)'},
                        title='기간 영업이익률')
    margin_fig.update_layout(showlegend=False)
    low_left, low_right = st.columns(2)
    low_left.plotly_chart(month_fig)
    low_right.plotly_chart(margin_fig)
```

In [ ]:
# C10 | 앱 검사: 노트북에서 앱을 실행하고 위젯을 바꿔 카드 값을 꺼냅니다.
from pathlib import Path
from streamlit.testing.v1 import AppTest

app_file = Path('ch11_app/app.py').resolve()     # 절대 경로
at = AppTest.from_file(str(app_file), default_timeout=10).run()
print([m.value for m in at.metric])
at.multiselect[0].set_value(['B']).run()
at.slider[0].set_value((10, 10)).run()
print([m.value for m in at.metric])